# VBZ Geo-Daten — Übersicht & Benchmark

Ausgangspunkt für den Visualisierungs-Benchmark: Tramnetz Zürich mit Stadtkreisen, Haltestellen und Verspätungs-Hotspots.

| # | Visualisierung | Beschreibung |
| :- | :--- | :--- |
| 1 | **Netzübersicht** | Stadtkreise als Polygone + Tramlinien in VBZ-Farben + Haltestellen |
| 2 | **Verspätungs-Heatmap** | Dichte-Darstellung von Verspätungen an den Haltestellen |
| 3 | **Streckenabschnitte** | Linienabschnitte eingefärbt nach Verspätung (Hotspot-Analyse) |

Jede Bibliothek hat ein eigenes Notebook:

- [`vbz-geo-map-plotly.ipynb`](vbz-geo-map-plotly.ipynb) — Plotly / Maplibre
- [`vbz-geo-map-geopandas.ipynb`](vbz-geo-map-geopandas.ipynb) — GeoPandas / Matplotlib
- [`vbz-geo-map-folium.ipynb`](vbz-geo-map-folium.ipynb) — Folium / Leaflet
- [`vbz-geo-map-kepler.ipynb`](vbz-geo-map-kepler.ipynb) — lonboard / Deck.gl

## Datenquellen

| Datensatz | Quelle | Format | Pfad |
| :--- | :--- | :--- | :--- |
| GTFS Tramlinien (Shapes) | VBZ / GTFS 2023–2025 | Parquet | `data/interim/vbz/gtfs/gtfs_tram_shapes.parquet` |
| GTFS Haltestellen | VBZ / GTFS 2023–2025 | Parquet | `data/interim/vbz/gtfs/gtfs_tram_stops.parquet` |
| GTFS Routen + Farben | VBZ / GTFS 2023–2025 | Parquet | `data/interim/vbz/gtfs/gtfs_tram_routes.parquet` |
| GTFS Fahrten | VBZ / GTFS 2023–2025 | Parquet | `data/interim/vbz/gtfs/gtfs_tram_trips.parquet` |
| Stadtkreis-Polygone | Stadt Zürich OGD (CC0) | GeoJSON WGS84 | `data/raw/vbz/stadtkreise/data/stzh_adm_stadtkreise_v.json` |
| Stadtkreis-Labels | Stadt Zürich OGD (CC0) | GeoJSON WGS84 | `data/raw/vbz/stadtkreise/data/stzh_adm_stadtkreise_beschr_p.json` |
| Ist-Daten (Verspätungen) | VBZ Soll-Ist-Vergleich | — | *noch nicht verfügbar — synthetische Daten als Platzhalter* |

**Lizenz Stadtkreise:** CC0 — freie Nutzung, Quellenangabe empfohlen: *Quelle: Stadt Zürich*

**Links:**
- OGD-Katalog Stadtkreise: https://data.stadt-zuerich.ch/dataset/geo_stadtkreise
- Referenz-Notebook Geoportal: https://github.com/opendatazurich/starter-code/blob/main/02_python/geo_stadtkreise_3d59fd6e-3e68-428e-8657-ed7acb042482.ipynb
- WFS-Dienst OGD Zürich: https://opendatazurich.github.io/geoportal/

In [8]:
import sys, os 
sys.path.append(os.path.abspath('../../../src'))
from doc_loader import show_doc


import pandas as pd
import numpy as np
import json
from pathlib import Path

# Repo-Root finden, egal von wo Jupyter gestartet wurde
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / 'data' / 'interim').exists():
        _ROOT = _p
        break

GTFS_DIR = _ROOT / 'data' / 'interim' / 'vbz' / 'gtfs'
GEO_DIR  = _ROOT / 'data' / 'raw' / 'vbz' / 'geo' / 'data'

---

## 1 — GTFS-Daten laden

Die GTFS-Parquet-Dateien wurden aus den offiziellen GTFS-Feeds der VBZ aufbereitet.
Enthalten sind die Jahre 2023, 2024 und 2025. **Referenzjahr: 2024.**

In [9]:
# Alle GTFS-Parquet-Dateien laden
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
routes = pd.read_parquet(GTFS_DIR / 'gtfs_tram_routes.parquet')
stops  = pd.read_parquet(GTFS_DIR / 'gtfs_tram_stops.parquet')
shapes = pd.read_parquet(GTFS_DIR / 'gtfs_tram_shapes.parquet')
trips  = pd.read_parquet(GTFS_DIR / 'gtfs_tram_trips.parquet')

print(f'Routen:  {len(routes):>6,}  →  {routes.columns.tolist()}')
print(f'Stops:   {len(stops):>6,}  →  {stops.columns.tolist()}')
print(f'Shapes:  {len(shapes):>6,}  →  {shapes.columns.tolist()}')
print(f'Trips:   {len(trips):>6,}  →  {trips.columns.tolist()}')
print(f'\nJahre in Daten: {sorted(routes["year"].unique())}')

Routen:      49  →  ['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type', 'route_color', 'route_text_color', 'year']
Stops:    7,202  →  ['stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'stop_url', 'location_type', 'parent_station', 'year']
Shapes:  535,930  →  ['shape_id', 'shape_pt_lat', 'shape_pt_lon', 'shape_pt_sequence', 'shape_dist_traveled', 'year']
Trips:   182,976  →  ['route_id', 'service_id', 'trip_id', 'shape_id', 'trip_headsign', 'direction_id', 'block_id', 'year']

Jahre in Daten: ['2023', '2024', '2025']


### Tramlinien 2024 mit VBZ-Farben

In [10]:
# Tramlinien 2024 mit offiziellen VBZ-Farben
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
routes_2024 = routes[routes['year'] == '2024'].drop_duplicates(subset=['route_short_name'])

print('Tramlinien 2024 mit VBZ-Farben:')
print(routes_2024[['route_short_name', 'route_long_name', 'route_color', 'route_text_color']]
      .sort_values('route_short_name')
      .to_string(index=False))

Tramlinien 2024 mit VBZ-Farben:
route_short_name  route_long_name route_color route_text_color
              10              NaN      E12472           FFFFFF
              11              NaN      00892F           FFFFFF
              12              NaN      92D6E3           000000
              13              NaN      FFCC00           000000
              14              NaN      008DC5           FFFFFF
              15              NaN      E20A16           FFFFFF
              17              NaN      8E224D           FFFFFF
              19              NaN      FFFFFF           000000
               2              NaN      E20A16           FFFFFF
               3              NaN      00892F           FFFFFF
               4              NaN      11296F           FFFFFF
               5              NaN      734522           FFFFFF
               6              NaN      CA7D3C           FFFFFF
               7              NaN      000000           FFFFFF
               8       

### Haltestellen 2024

In [11]:
# Haltestellen 2024 — Koordinaten und Abdeckung
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
stops_2024 = stops[stops['year'] == '2024'].copy()

print(f'Haltestellen gesamt (2024): {len(stops_2024):,}')
print(f'Eindeutige Stop-Namen:      {stops_2024["stop_name"].nunique():,}')
print(f'Koordinaten-Bereich:')
print(f'  Lat: {stops_2024["stop_lat"].min():.4f} – {stops_2024["stop_lat"].max():.4f}')
print(f'  Lon: {stops_2024["stop_lon"].min():.4f} – {stops_2024["stop_lon"].max():.4f}')
print()
print(stops_2024.head(5)[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']])

Haltestellen gesamt (2024): 2,446
Eindeutige Stop-Namen:      1,189
Koordinaten-Bereich:
  Lat: 47.3002 – 47.4500
  Lon: 8.4507 – 8.6499

                   stop_id                  stop_name   stop_lat  stop_lon
2379  ch:1:sloid:10010::20  Zürich, Klinik Hirslanden  47.351897  8.576602
2380  ch:1:sloid:10010::21  Zürich, Klinik Hirslanden  47.351094  8.576890
2381  ch:1:sloid:10011::50       Zürich, Kinderspital  47.352603  8.572254
2382  ch:1:sloid:10011::51       Zürich, Kinderspital  47.352561  8.572362
2383  ch:1:sloid:10012::50   Zollikon, Rotfluhstrasse  47.343303  8.576980


### Shape-Statistiken

In [12]:
# Shape-Punkte 2024 — Übersicht
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
shapes_2024 = shapes[shapes['year'] == '2024'].copy()

pts_per_shape = shapes_2024.groupby('shape_id').size()

print(f'Shape-Punkte gesamt (2024): {len(shapes_2024):,}')
print(f'Eindeutige Shape-IDs:       {shapes_2024["shape_id"].nunique():,}')
print(f'Punkte pro Shape:  Ø {pts_per_shape.mean():.0f},  Min {pts_per_shape.min()},  Max {pts_per_shape.max()}')

Shape-Punkte gesamt (2024): 171,754
Eindeutige Shape-IDs:       582
Punkte pro Shape:  Ø 295,  Min 2,  Max 641


---

## 2 — Stadtkreis-GeoJSON 

### Datenquellen

Verweis auf vbz-data

### Lokale Pfade

Verweis auf vbz-data

### Datenwörterbuch

Verweis auf vbz-data

In [13]:
show_doc('geo')

## Datenwörterbuch — Stadtkreise Zürich (GeoJSON)

Dieses Wörterbuch beschreibt die Attribute der geografischen Grenzen der 12 Zürcher Stadtkreise. Diese Daten sind essenziell für räumliche Analysen und Visualisierungen (Choroplethenkarten).

| Attribut | Datentyp | Beschreibung | Beispiel |
| :--- | :--- | :--- | :--- |
| **knr** | Integer | **Kreisnummer:** Eindeutiger Identifikator für den Stadtkreis (1 bis 12). | `9` |
| **kname** | String | **Kreisname:** Offizielle Bezeichnung des Stadtkreises. | `"Kreis 9"` |
| **objid** | Integer | **Objekt-ID:** Interne Kennung des Datensatzes im GIS-System der Stadt Zürich. | `42` |
| **geometry** | Polygon | **Geometrie:** Enthält die Koordinatenpunkte (WGS84 oder LV95), die die Grenze des Kreises definieren. | `[[[268..., 124...], ...]]` |

**Hinweise zur Verwendung:**
* **Join-Key:** Die Spalte `knr` ist der primäre Schlüssel, um Wetter- oder Tramdaten auf Kreisebene zu aggregieren.
* **Hierarchie:** Ein Stadtkreis besteht aus mehreren statistischen Quartieren. Für feinere Analysen sollte der Datensatz "Statistische Quartiere" verwendet werden.
* **Visualisierung:** In Python (GeoPandas) wird die Spalte `geometry` automatisch erkannt, um Karten zu plotten.

In [14]:
# Stadtkreis-GeoJSON laden
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
with open(GEO_DIR / 'stzh_adm_stadtkreise_v.json') as f:
    stadtkreise_geo = json.load(f)

with open(GEO_DIR / 'stzh_adm_stadtkreise_beschr_p.json') as f:
    labels_geo = json.load(f)

print(f'Stadtkreis-Features: {len(stadtkreise_geo["features"])}')
print(f'Label-Features:      {len(labels_geo["features"])}')
print(f'CRS: {stadtkreise_geo.get("crs", "nicht angegeben — WGS84 angenommen")}')
print()

for f in stadtkreise_geo['features']:
    p = f['properties']
    n_coords = len(f['geometry']['coordinates'][0])
    print(f'  {p["kname"]:10s} (Kreis {p["knr"]:>2})  |  {f["geometry"]["type"]}  |  {n_coords} Stützpunkte')

Stadtkreis-Features: 12
Label-Features:      12
CRS: nicht angegeben — WGS84 angenommen

  Kreis 6    (Kreis  6)  |  Polygon  |  1600 Stützpunkte
  Kreis 11   (Kreis 11)  |  Polygon  |  1567 Stützpunkte
  Kreis 12   (Kreis 12)  |  Polygon  |  700 Stützpunkte
  Kreis 10   (Kreis 10)  |  Polygon  |  1229 Stützpunkte
  Kreis 4    (Kreis  4)  |  Polygon  |  717 Stützpunkte
  Kreis 1    (Kreis  1)  |  Polygon  |  508 Stützpunkte
  Kreis 9    (Kreis  9)  |  Polygon  |  1632 Stützpunkte
  Kreis 5    (Kreis  5)  |  Polygon  |  469 Stützpunkte
  Kreis 7    (Kreis  7)  |  Polygon  |  2460 Stützpunkte
  Kreis 3    (Kreis  3)  |  Polygon  |  1138 Stützpunkte
  Kreis 2    (Kreis  2)  |  Polygon  |  850 Stützpunkte
  Kreis 8    (Kreis  8)  |  Polygon  |  894 Stützpunkte


---

## 3 — Synthetische Verspätungsdaten (Platzhalter)

Echte Ist-Daten (Soll-Ist-Vergleich auf Fahrtebene) sind noch nicht in den aufbereiteten Datensätzen enthalten. Die Visualisierungs-Notebooks arbeiten daher mit **synthetischen Verspätungsdaten** als Platzhalter:

```python
np.random.seed(42)
stops_unique['delay_min'] = np.random.exponential(scale=1.5, size=len(stops_unique)).round(1)
```

Die Werte sind exponentialverteilt (Median ≈ 1 Min, langes Tail für Hotspots) — ähnlich dem Muster realer ÖPNV-Verspätungen.

Sobald die Ist-Daten verfügbar sind, können die `delay_min`-Spalten direkt ersetzt werden — die Visualisierungslogik bleibt unverändert.

In [15]:
# Synthetische Verspätungsdaten — Platzhalter bis Ist-Daten verfügbar
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
np.random.seed(42)

stops_unique = stops_2024.drop_duplicates(subset=['stop_name']).copy()
stops_unique['delay_min'] = np.random.exponential(scale=1.5, size=len(stops_unique)).round(1)

print(f'Haltestellen mit Delay-Werten: {len(stops_unique):,}')
print(f'Verspätung Min / Median / Max: '
      f'{stops_unique["delay_min"].min():.1f} / '
      f'{stops_unique["delay_min"].median():.1f} / '
      f'{stops_unique["delay_min"].max():.1f} Min')
print()

print('Top-10 Verspätungs-Hotspots (synthetisch):')
print(stops_unique.nlargest(10, 'delay_min')[['stop_name', 'delay_min', 'stop_lat', 'stop_lon']].to_string(index=False))

Haltestellen mit Delay-Werten: 1,189
Verspätung Min / Median / Max: 0.0 / 1.1 / 12.3 Min

Top-10 Verspätungs-Hotspots (synthetisch):
                     stop_name  delay_min  stop_lat  stop_lon
        Zürich, Bucheggplatz E       12.3 47.398496  8.533151
                 Dangelstrasse        9.3 47.334537  8.529962
            Zürich, Werdhölzli        8.7 47.397790  8.482055
        Zürich, Bucheggplatz A        8.5 47.398319  8.533375
                   Limmatplatz        7.9 47.384599  8.531624
         Zürich, Bad Allenmoos        7.4 47.405950  8.537642
                   Hönggerberg        7.1 47.405032  8.504818
       Regensdorf, Querstrasse        7.0 47.444658  8.462094
D'dorf, Sportanl Heerenschürli        6.9 47.404339  8.595835
   Zürich, Messe/Hallenstadion        6.9 47.411008  8.550955


---

## 4 — Visualisierungs-Benchmark

### Bewertungsmatrix

| Kriterium | Plotly | GeoPandas | Folium | lonboard |
| :--- | :---: | :---: | :---: | :---: |
| **VS Code rendering** | ✅ | ✅ | ⚠️ (HTML export) | ⚠️ (HTML export) |
| **Interaktivität** | ✅ | ❌ | ✅ | ✅ |
| **Tramlinien (VBZ-Farben)** | ✅ | ✅ | ✅ | ✅ |
| **Haltestellen** | ✅ | ✅ | ✅ | ✅ |
| **Stadtkreis-Polygone** | ✅ | ✅ | ✅ | ✅ |
| **Heatmap** | ✅ | ✅ | ✅ | ✅ |
| **Streckenabschnitte farbig** | ✅ | ✅ | ✅ | ✅ |
| **Dashboard-Integration** | ✅ (Dash) | ❌ | ❌ | ⚠️ (custom) |
| **PDF / Print-Export** | ⚠️ | ✅ | ❌ | ❌ |
| **Python 3.13 Support** | ✅ | ✅ | ✅ | ✅ |
| **Setup-Aufwand** | Gering | Mittel | Gering | Hoch |
| **Code-Komplexität** | Mittel | Mittel | Gering | Gering |
| **Performance (535k Punkte)** | Gut | Gut | Langsam | Sehr gut (GPU) |

---

### Empfehlungen nach Anwendungsfall

| Anwendungsfall | Empfehlung | Begründung |
| :--- | :--- | :--- |
| EDA / Prototyping | **Plotly** | VS Code nativ, schnell, kein Browser nötig |
| Report / Druck | **GeoPandas** | PDF/SVG-Export, Druckqualität |
| Dashboard | **Plotly + Dash** | Callbacks, Filter, Deployment |
| Tiefenanalyse | **lonboard** | GPU-Rendering (Deck.gl), Python 3.13 kompatibel |
| Einfacher Austausch | **Folium** | HTML-Datei, kein Python zum Anschauen nötig |

---

#### EDA & schnelles Prototyping → Plotly oder Folium

Beide liefern sofort interaktive Karten. Plotly läuft direkt in VS Code ohne Browser-Umweg; Folium generiert HTML — gut für ad-hoc Exploration und Weitergabe.

#### Statischer Report / Präsentation → GeoPandas + Matplotlib

Einzige Bibliothek mit sauberem Vektorexport (PDF, SVG). Pixel-präzise Kontrolle über Schriften, Farben und Layout. Ideal für LaTeX- oder Word-Berichte.

#### Operatives Dashboard → Plotly + Dash

Plotly-Karten lassen sich nahtlos in Dash-Dashboards einbetten. Filter, Zeitachsen und Callbacks direkt integrierbar.

#### Explorative Tiefenanalyse → lonboard (Deck.gl, GPU)

GPU-beschleunigtes Rendering: HeatmapLayer, PathLayer, ScatterplotLayer ohne Trace-Limit. Python 3.13 kompatibel — Ersatz für keplergl, das wegen `pyarrow~=16.0.0` nicht mit Python 3.13 baut.